In [0]:
# Reads from bronze.assignments
spark.sql("USE CATALOG workspace")

from pyspark.sql.functions import (
    col, trim, lit, when, current_timestamp,
    datediff, count
)
from pyspark.sql.window import Window

#read from bronze and silver reference tables
bronze_df     = spark.table("bronze.assignments")
silver_emp_df = spark.table("silver.employees").select("employee_id")
silver_prj_df = spark.table("silver.projects").select("project_id")

print(f"Bronze assignments: {bronze_df.count()}")
print(f"Silver employees:   {silver_emp_df.count()}")
print(f"Silver projects:    {silver_prj_df.count()}")
bronze_df.printSchema()

In [0]:

from pyspark.sql.functions import lag
from pyspark.sql.window import Window

# validate employee and project IDs exist in silver
valid_emp_ids = [row.employee_id for row in silver_emp_df.collect()]
valid_prj_ids = [row.project_id for row in silver_prj_df.collect()]

cleaned_df = (
    bronze_df
    # Trim whitespace
    .withColumn("role",          trim(col("role")))
    .withColumn("practice_area", trim(col("practice_area")))

    # Validate date logic
    .withColumn("date_valid", when(
        col("end_date") > col("start_date"),
        lit(True)).otherwise(lit(False)))

    #calculate assignment duration
    .withColumn("duration_days", datediff(col("end_date"), col("start_date")))

 
    .withColumn("employee_valid", when(
        col("employee_id").isin(valid_emp_ids),
        lit(True)).otherwise(lit(False)))

    .withColumn("project_valid", when(
        col("project_id").isin(valid_prj_ids),
        lit(True)).otherwise(lit(False)))

    #flag nulls in critical fields
    .withColumn("has_nulls", when(
        col("assignment_id").isNull() |
        col("employee_id").isNull() |
        col("project_id").isNull() |
        col("start_date").isNull() |
        col("end_date").isNull(),
        lit(True)).otherwise(lit(False)))

 
    .withColumn("silver_loaded_at", current_timestamp())
    .withColumn("source_table",     lit("bronze.assignments"))
)


cleaned_df = cleaned_df.dropDuplicates(["assignment_id"])

print(f"Silver row count:       {cleaned_df.count()}")
print(f"Invalid dates:          {cleaned_df.filter(col('date_valid') == False).count()}")
print(f"Invalid employee IDs:   {cleaned_df.filter(col('employee_valid') == False).count()}")
print(f"Invalid project IDs:    {cleaned_df.filter(col('project_valid') == False).count()}")
print(f"Rows with nulls:        {cleaned_df.filter(col('has_nulls') == True).count()}")
cleaned_df.show(5)

In [0]:
(
    cleaned_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.assignments")
)

print(f"Written to silver.assignments: {spark.table('silver.assignments').count()} rows")